# Paper 1 — Golden Age Semantic Reconfiguration

**Working title:** *Reconfiguring the Golden Age: Semantic Networks and the Renaissance–Baroque Transition in Spanish Poetry*

This is the **single working notebook** for Paper 1. GitHub is the source of truth.

> **Current stage:** Sprint 1 — corpus audit, corpus reconciliation and temporal reconstruction.  
> We do **not** build semantic networks until the corpus and temporal backbone are defensible.


## Colab ↔ GitHub workflow

1. Open this notebook directly from `ardominguezm/golden-age-semantic-reconfiguration`.
2. Run **Runtime → Run all**.
3. Inspect the scientific checkpoints.
4. Preserve the run with **File → Save a copy in GitHub**, overwriting this same file on `main`.
5. Do not create parallel experimental notebooks.

Each code update may temporarily clear outputs; the previous executed state remains preserved in Git history.


## 00. Environment & reproducibility

The two upstream corpora are public and pinned to exact commits. This prevents future upstream changes from silently changing our results.


In [ ]:
import sys, os, re, shutil, subprocess, unicodedata, math
from pathlib import Path
from collections import Counter, defaultdict
from difflib import SequenceMatcher
import pandas as pd
import xml.etree.ElementTree as ET

IN_COLAB = "google.colab" in sys.modules
print(f"Running in Colab: {IN_COLAB}")
print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__}")


## 01. Authoritative sources

We audit two complementary sources:

- **Hernández-Lorenzo network corpus**: the derivative corpus used in the previous network study, with standardized orthography and author-level metadata.
- **CorpusSonetosSigloDeOro (Navarro Colorado)**: individual TEI/XML sonnets with titles, textual structure and bibliographic source information.

The current question is whether the Hernández aggregate TXT files can be reliably split back into poems and reconciled with the TEI corpus.


In [ ]:
SOURCES = {
    "hernandez_network": {
        "repo": "https://github.com/lamusadecima/Network_for_Golden_Age_Spanish_Poetry.git",
        "commit": "ef6b7b691f67abe60d9cfa85c274f0be8095dd9a",
    },
    "navarro_tei": {
        "repo": "https://github.com/bncolorado/CorpusSonetosSigloDeOro.git",
        "commit": "092a5fe70a4065a4d84bfed288bffd3851348f9c",
    },
}

SOURCE_ROOT = Path("/content/gasr_sources")
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)

def clone_at_commit(name, repo_url, commit):
    target = SOURCE_ROOT / name
    if target.exists():
        shutil.rmtree(target)
    subprocess.run(["git","clone","--quiet",repo_url,str(target)], check=True)
    subprocess.run(["git","-C",str(target),"checkout","--quiet",commit], check=True)
    resolved = subprocess.check_output(
        ["git","-C",str(target),"rev-parse","HEAD"], text=True
    ).strip()
    assert resolved == commit, (name, resolved, commit)
    return target

source_paths = {
    name: clone_at_commit(name, info["repo"], info["commit"])
    for name, info in SOURCES.items()
}

print("Pinned sources ready:")
for name, path in source_paths.items():
    print(f"  {name}: {path}")


## 02. Hernández-Lorenzo metadata: what does `Date` mean?

The previous corpus contains `corpus/metadata.csv`. We explicitly test whether `Date` is poem chronology or biographical metadata.


In [ ]:
hernandez_root = source_paths["hernandez_network"]
navarro_root = source_paths["navarro_tei"]

h_meta = pd.read_csv(hernandez_root / "corpus" / "metadata.csv")
h_meta = h_meta.loc[:, ~h_meta.columns.astype(str).str.startswith("Unnamed")].copy()

life_pat = re.compile(r"^\s*(\d{4})\s*[-–]\s*(\d{4})\s*$")

def parse_life(x):
    m = life_pat.match(str(x))
    return (int(m.group(1)), int(m.group(2))) if m else (pd.NA, pd.NA)

life = h_meta["Date"].apply(parse_life)
h_meta["birth_year"] = [x[0] for x in life]
h_meta["death_year"] = [x[1] for x in life]

print(f"Metadata rows: {len(h_meta)}")
print(f"Rows whose Date field is YYYY-YYYY: {h_meta['birth_year'].notna().sum()}/{len(h_meta)}")
print(f"Sum of metadata poem counts: {int(h_meta['Poems'].sum()):,}")
display(h_meta.head(10))

print("\nConclusion: Hernández `Date` is author lifespan metadata, not poem-level dating.")


## 03. Recover poem boundaries from Hernández aggregate TXT files

The TXT files usually separate sonnets with blank lines. We recover blocks and audit line counts rather than assuming every block is valid.


In [ ]:
txt_files = sorted((hernandez_root / "corpus").glob("*_Sonetos*.txt"))
# Include files such as AN_SonetosP2.txt that follow the same structure.
txt_files = sorted(set(txt_files) | set((hernandez_root / "corpus").glob("*Sonetos*.txt")))

def split_blank_blocks(text):
    lines = text.splitlines()
    blocks, current = [], []
    for line in lines:
        if line.strip():
            current.append(line.strip())
        else:
            if current:
                blocks.append(current)
                current = []
    if current:
        blocks.append(current)
    return blocks

h_rows = []
for p in txt_files:
    author_file = re.sub(r"_Sonetos.*$", "", p.stem)
    blocks = split_blank_blocks(p.read_text(encoding="utf-8", errors="replace"))
    for i, block in enumerate(blocks, start=1):
        h_rows.append({
            "h_id": f"{author_file}_{i:04d}",
            "source_file": p.name,
            "author_file": author_file,
            "poem_local_id": i,
            "n_lines": len(block),
            "text": "\n".join(block),
            "first_line": block[0] if block else "",
            "last_line": block[-1] if block else "",
        })

h_poems = pd.DataFrame(h_rows)

print(f"TXT author files: {len(txt_files)}")
print(f"Recovered poem blocks: {len(h_poems):,}")
print(f"14-line blocks: {(h_poems.n_lines == 14).sum():,} / {len(h_poems):,}")
print("\nLine-count distribution:")
print(h_poems["n_lines"].value_counts().sort_index().to_string())

h_author_counts = (
    h_poems.groupby(["source_file","author_file"])
    .agg(recovered_poems=("h_id","count"),
         blocks_14_lines=("n_lines", lambda s: int((s==14).sum())))
    .reset_index()
    .sort_values("recovered_poems", ascending=False)
)
display(h_author_counts.head(20))


### 03.1 Sanity checks against reported author totals

Blank-line splitting is only a candidate segmentation. We test several known authors and flag differences.


In [ ]:
def norm_letters(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return re.sub(r"[^a-z]", "", s.lower())

checks = {
    "Cervantes": ("Cervantes, Miguel de",),
    "Garcilaso": ("Vega, Garcilaso de la",),
    "Gongora": ("Góngora, Luis de",),
    "Quevedo": ("Quevedo, Francisco de",),
    "Herrera": ("Herrera, Fernando de",),
    "Pacheco": ("Pacheco, Francisco",),
}

rows = []
for file_key, author_candidates in checks.items():
    file_rows = h_author_counts[
        h_author_counts["author_file"].map(norm_letters).str.contains(norm_letters(file_key), regex=False)
    ]
    meta_rows = h_meta[h_meta["Author"].isin(author_candidates)]
    recovered = int(file_rows["recovered_poems"].sum()) if len(file_rows) else pd.NA
    expected = int(meta_rows["Poems"].sum()) if len(meta_rows) else pd.NA
    rows.append({
        "file_key": file_key,
        "recovered": recovered,
        "metadata_expected": expected,
        "match": (recovered == expected) if pd.notna(recovered) and pd.notna(expected) else pd.NA
    })

sanity = pd.DataFrame(rows)
display(sanity)

print("\nImportant: mismatches are segmentation diagnostics, not missing poems until verified against TEI/text.")


## 04. Parse Navarro TEI at poem level

We use a conservative TEI parser. For lines containing textual variants, the lemma (`<lem>`) is preferred so that a single canonical line is represented.


In [ ]:
NS = {"tei": "http://www.tei-c.org/ns/1.0"}
xml_files = sorted(navarro_root.glob("*/*.xml"))

def localname(tag):
    return tag.split("}",1)[-1] if "}" in tag else tag

def element_text(el):
    return " ".join("".join(el.itertext()).split()) if el is not None else ""

def canonical_line_text(line_el):
    lem = line_el.find(".//tei:lem", NS)
    if lem is not None:
        return element_text(lem)
    # If there is no apparatus, ordinary itertext is safe.
    if line_el.find(".//tei:app", NS) is None:
        return element_text(line_el)
    # Fallback: prefer all non-rdg text fragments.
    pieces = []
    if line_el.text:
        pieces.append(line_el.text)
    for child in list(line_el):
        if localname(child.tag) == "rdg":
            if child.tail:
                pieces.append(child.tail)
            continue
        pieces.extend(child.itertext())
        if child.tail:
            pieces.append(child.tail)
    return " ".join(" ".join(pieces).split())

n_rows, parse_errors = [], []
for p in xml_files:
    try:
        root = ET.parse(p).getroot()
        title_el = root.find(".//tei:text/tei:body/tei:head/tei:title", NS)
        author_el = root.find(".//tei:sourceDesc//tei:author", NS)
        bibl_el = root.find(".//tei:sourceDesc//tei:bibl", NS)
        line_els = root.findall(".//tei:text/tei:body//tei:l", NS)
        lines = [canonical_line_text(el) for el in line_els]
        n_rows.append({
            "n_id": str(p.relative_to(navarro_root)).replace("/", "::").replace(".xml",""),
            "source_file": str(p.relative_to(navarro_root)),
            "author_dir": p.parent.name,
            "author_tei": element_text(author_el),
            "title": element_text(title_el),
            "n_lines": len(lines),
            "text": "\n".join(lines),
            "first_line": lines[0] if lines else "",
            "last_line": lines[-1] if lines else "",
            "source_bibl": element_text(bibl_el),
        })
    except Exception as exc:
        parse_errors.append((str(p.relative_to(navarro_root)), repr(exc)))

n_poems = pd.DataFrame(n_rows)

print(f"Parsed TEI poems: {len(n_poems):,}")
print(f"Parse errors: {len(parse_errors):,}")
print(f"Author folders: {n_poems['author_dir'].nunique():,}")
print(f"14-line records: {(n_poems.n_lines == 14).sum():,} / {len(n_poems):,}")
display(n_poems[["n_id","author_dir","author_tei","title","n_lines"]].head(10))


## 05. Exact TEI temporal audit

We search XML structure, not arbitrary substrings. Dates found under witnesses/editions are kept as bibliographic evidence and are **not** converted into composition dates.


In [ ]:
TEMP_ATTRS = {"when","notbefore","notafter","from","to"}
date_rows = []

def walk_with_context(el, ancestors, file_rel):
    lname = localname(el.tag).lower()

    if lname == "date":
        value = element_text(el)
        if value:
            date_rows.append({
                "file": file_rel,
                "kind": "date_element",
                "field": "date",
                "value": value,
                "element": lname,
                "context": ">".join(reversed(ancestors[-5:])),
            })

    for k, v in el.attrib.items():
        ak = localname(k).lower()
        if ak in TEMP_ATTRS:
            date_rows.append({
                "file": file_rel,
                "kind": "date_attribute",
                "field": ak,
                "value": str(v),
                "element": lname,
                "context": ">".join(reversed(ancestors[-5:])),
            })

    for child in list(el):
        walk_with_context(child, ancestors + [lname], file_rel)

for p in xml_files:
    root = ET.parse(p).getroot()
    walk_with_context(root, [], str(p.relative_to(navarro_root)))

tei_dates = pd.DataFrame(date_rows)

if len(tei_dates):
    print(f"Exact TEI temporal records: {len(tei_dates):,}")
    print(f"Files with exact temporal records: {tei_dates['file'].nunique():,}")
    display(tei_dates)
else:
    print("No exact TEI date elements/attributes found.")


In [ ]:
def classify_tei_date(row):
    ctx = str(row["context"]).lower()
    if "witness" in ctx or "listwit" in ctx:
        return "witness_or_edition_date"
    if "sourcedesc" in ctx or "bibl" in ctx:
        return "bibliographic_date"
    return "unclassified_temporal_record"

if len(tei_dates):
    tei_dates["date_role"] = tei_dates.apply(classify_tei_date, axis=1)
    display(
        tei_dates.groupby("date_role")
        .agg(records=("file","count"), files=("file","nunique"))
        .reset_index()
    )
    print("\nThese dates are not automatically poem composition dates.")


## 06. Author-level reconciliation

Before poem matching, we inspect author availability. Of particular interest are Herrera and Pacheco.


In [ ]:
h_author_labels = sorted(h_author_counts["author_file"].astype(str).unique())
n_author_labels = sorted(n_poems["author_dir"].astype(str).unique())

def contains_author(labels, key):
    nk = norm_letters(key)
    return [x for x in labels if nk in norm_letters(x)]

targets = ["Herrera","Pacheco","Garcilaso","Gongora","Quevedo","Cervantes","Lope"]
target_rows = []
for key in targets:
    target_rows.append({
        "target": key,
        "hernandez_matches": ", ".join(contains_author(h_author_labels, key)),
        "navarro_matches": ", ".join(contains_author(n_author_labels, key)),
    })
display(pd.DataFrame(target_rows))

print(f"Hernández author-file labels: {len(h_author_labels)}")
print(f"Navarro author folders: {len(n_author_labels)}")


# 07. NEW — Poem-level corpus reconciliation by text signatures

This is the next decisive test.

We now ask:

1. How much of the Hernández corpus can be linked automatically to individual Navarro TEI poems?
2. Are the count mismatches (e.g. Herrera/Quevedo) real omissions or merely segmentation artefacts?
3. Can Navarro serve as the poem-level backbone while Hernández contributes a standardized textual representation and source-exclusive material such as Pacheco?

We use two stages:

- **exact normalized-text match**;
- **high-similarity fuzzy match within the most plausible author**, only for records not exactly matched.

Fuzzy matches are diagnostics until a conservative threshold is chosen.


In [ ]:
# RapidFuzz is fast enough for author-restricted matching in Colab.
try:
    from rapidfuzz import fuzz, process
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz"], check=True)
    from rapidfuzz import fuzz, process

def normalize_text(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.lower()
    # retain letters only: punctuation, whitespace and capitalization do not affect matching
    return re.sub(r"[^a-zñ]", "", s)

h_poems["signature"] = h_poems["text"].map(normalize_text)
n_poems["signature"] = n_poems["text"].map(normalize_text)

# Global exact-match index. Keep only unique signatures on the TEI side.
sig_to_nids = defaultdict(list)
for row in n_poems[["n_id","signature"]].itertuples(index=False):
    sig_to_nids[row.signature].append(row.n_id)

def exact_target(sig):
    ids = sig_to_nids.get(sig, [])
    return ids[0] if len(ids) == 1 else pd.NA

h_poems["exact_n_id"] = h_poems["signature"].map(exact_target)
h_poems["exact_unique_match"] = h_poems["exact_n_id"].notna()

exact_n = int(h_poems["exact_unique_match"].sum())
print(f"Unique exact poem matches: {exact_n:,} / {len(h_poems):,} ({exact_n/len(h_poems):.1%})")

exact_join = (
    h_poems[h_poems["exact_unique_match"]]
    [["h_id","author_file","exact_n_id","n_lines"]]
    .merge(
        n_poems[["n_id","author_dir","title","n_lines"]],
        left_on="exact_n_id", right_on="n_id", how="left",
        suffixes=("_h","_n")
    )
)

author_exact = (
    exact_join.groupby(["author_file","author_dir"])
    .size().reset_index(name="exact_matches")
    .sort_values(["author_file","exact_matches"], ascending=[True,False])
)

print("\nTop inferred author correspondences from exact poem matches:")
display(author_exact.groupby("author_file", as_index=False).head(1).head(40))


### 07.1 Infer author correspondences conservatively

Exact poem matches give us empirical author mappings where available. For authors without exact evidence we use name similarity only as a diagnostic, and we never auto-map Pacheco to an unrelated author.


In [ ]:
# Dominant exact-based mapping per Hernández author.
dominant_exact = (
    author_exact.sort_values(["author_file","exact_matches"], ascending=[True,False])
    .drop_duplicates("author_file")
    .set_index("author_file")["author_dir"]
    .to_dict()
)

def best_name_author(h_label):
    scored = []
    nh = norm_letters(h_label)
    for n_label in n_author_labels:
        score = SequenceMatcher(None, nh, norm_letters(n_label)).ratio()
        scored.append((score, n_label))
    return max(scored) if scored else (0.0, None)

author_map_rows = []
for h_label in h_author_labels:
    if h_label in dominant_exact:
        mapped = dominant_exact[h_label]
        method = "exact_poem_evidence"
        confidence = 1.0
    elif norm_letters(h_label) == norm_letters("Pacheco"):
        mapped = None
        method = "source_exclusive_candidate"
        confidence = 1.0
    else:
        score, candidate = best_name_author(h_label)
        mapped = candidate
        method = "name_similarity_diagnostic"
        confidence = score
    author_map_rows.append({
        "author_file": h_label,
        "navarro_author_dir": mapped,
        "mapping_method": method,
        "mapping_confidence": confidence,
    })

author_map = pd.DataFrame(author_map_rows)
display(author_map.sort_values(["mapping_method","mapping_confidence"], ascending=[True,False]))

print("\nMappings without exact poem evidence:")
display(author_map[author_map["mapping_method"] != "exact_poem_evidence"])


### 07.2 High-similarity fuzzy matching for unmatched poems

For each unmatched Hernández block with a plausible Navarro author, we search only that author's TEI poems.

Interpretation thresholds used **for diagnostics**:

- `>= 98`: near-identical, very strong candidate;
- `95–97.99`: strong candidate;
- `90–94.99`: review;
- `< 90`: unresolved.

These thresholds are not yet final inclusion rules.


In [ ]:
# Build author-specific candidate dictionaries for RapidFuzz.
n_choices = {
    author: dict(zip(g["n_id"], g["signature"]))
    for author, g in n_poems.groupby("author_dir")
}

map_dict = author_map.set_index("author_file")["navarro_author_dir"].to_dict()

fuzzy_rows = []
unmatched_h = h_poems[~h_poems["exact_unique_match"]].copy()

for row in unmatched_h.itertuples(index=False):
    candidate_author = map_dict.get(row.author_file)
    if candidate_author is None or pd.isna(candidate_author) or candidate_author not in n_choices:
        fuzzy_rows.append({
            "h_id": row.h_id,
            "author_file": row.author_file,
            "candidate_author": candidate_author,
            "fuzzy_n_id": pd.NA,
            "fuzzy_score": pd.NA,
        })
        continue

    choices = n_choices[candidate_author]
    # process.extractOne returns (matched_value, score, matched_key) for dict choices.
    hit = process.extractOne(row.signature, choices, scorer=fuzz.ratio)
    if hit is None:
        fuzzy_n_id, score = pd.NA, pd.NA
    else:
        _, score, fuzzy_n_id = hit

    fuzzy_rows.append({
        "h_id": row.h_id,
        "author_file": row.author_file,
        "candidate_author": candidate_author,
        "fuzzy_n_id": fuzzy_n_id,
        "fuzzy_score": float(score) if pd.notna(score) else pd.NA,
    })

fuzzy = pd.DataFrame(fuzzy_rows)

def fuzzy_band(x):
    if pd.isna(x):
        return "no_candidate_author"
    if x >= 98:
        return "near_identical_98+"
    if x >= 95:
        return "strong_95_98"
    if x >= 90:
        return "review_90_95"
    return "unresolved_lt90"

fuzzy["band"] = fuzzy["fuzzy_score"].map(fuzzy_band)

print("Fuzzy-match diagnostic bands among non-exact Hernández blocks:")
display(
    fuzzy["band"].value_counts(dropna=False)
    .rename_axis("band").reset_index(name="records")
)

print("\nScore distribution:")
display(fuzzy["fuzzy_score"].describe().to_frame().T)


### 07.3 Author-specific diagnostics: Herrera, Pacheco, Quevedo and canonical controls


In [ ]:
diag = (
    h_poems[["h_id","author_file","n_lines","exact_unique_match"]]
    .merge(fuzzy[["h_id","fuzzy_score","band"]], on="h_id", how="left")
)

key_authors = ["Herrera","Pacheco","Quevedo","Cervantes","Garcilaso","Gongora"]
rows = []
for key in key_authors:
    sub = diag[diag["author_file"].map(norm_letters).str.contains(norm_letters(key), regex=False)]
    rows.append({
        "author_key": key,
        "h_blocks": len(sub),
        "h_14line_blocks": int((sub["n_lines"]==14).sum()),
        "exact_matches": int(sub["exact_unique_match"].sum()),
        "fuzzy_98plus": int((sub["fuzzy_score"]>=98).sum()),
        "fuzzy_95plus": int((sub["fuzzy_score"]>=95).sum()),
        "unresolved_lt90_or_no_candidate": int(((sub["fuzzy_score"]<90) | sub["fuzzy_score"].isna()).sum()),
    })

key_diag = pd.DataFrame(rows)
display(key_diag)

print("\nNon-14-line Hernández blocks (first 30):")
display(
    h_poems[h_poems["n_lines"] != 14]
    [["h_id","author_file","n_lines","first_line","last_line"]]
    .head(30)
)


## 08. Candidate master-corpus policy

We do **not** automatically union both corpora because unmatched records may be the same poem under orthographic/editorial variation.

The default candidate policy is:

1. **Navarro TEI as the poem-level backbone** because it has explicit poem boundaries, titles, line structure and bibliographic provenance.
2. Attach Hernández standardized text to Navarro poems only when matching confidence is high.
3. Add genuinely source-exclusive Hernández material only after author-level and poem-level validation — Pacheco is the main expected case.
4. Preserve both text representations (`text_tei`, `text_standardized`) rather than overwriting one with the other.
5. Keep all matching diagnostics so every inclusion decision is auditable.


In [ ]:
# Conservative linkage table:
# exact matches are accepted; fuzzy >=98 are marked as provisional high-confidence links.
link_exact = h_poems[h_poems["exact_unique_match"]][["h_id","author_file","exact_n_id","text"]].copy()
link_exact = link_exact.rename(columns={"exact_n_id":"n_id","text":"text_standardized"})
link_exact["link_method"] = "exact_normalized_text"
link_exact["link_score"] = 100.0

link_fuzzy = (
    fuzzy[fuzzy["fuzzy_score"] >= 98]
    .merge(h_poems[["h_id","text"]], on="h_id", how="left")
    [["h_id","author_file","fuzzy_n_id","text","fuzzy_score"]]
    .rename(columns={
        "fuzzy_n_id":"n_id",
        "text":"text_standardized",
        "fuzzy_score":"link_score"
    })
)
link_fuzzy["link_method"] = "fuzzy_98plus_provisional"

links = pd.concat([link_exact, link_fuzzy], ignore_index=True)

# Detect collisions: multiple Hernández records pointing to the same TEI poem.
collision_counts = links.groupby("n_id").size()
collision_nids = set(collision_counts[collision_counts > 1].index)
links["collision"] = links["n_id"].isin(collision_nids)

print(f"Accepted exact links: {(links.link_method=='exact_normalized_text').sum():,}")
print(f"Provisional fuzzy >=98 links: {(links.link_method=='fuzzy_98plus_provisional').sum():,}")
print(f"TEI targets with linkage collisions: {len(collision_nids):,}")

master_candidate = n_poems.rename(columns={"text":"text_tei"}).copy()
best_links = (
    links[~links["collision"]]
    .sort_values(["n_id","link_score"], ascending=[True,False])
    .drop_duplicates("n_id")
)
master_candidate = master_candidate.merge(
    best_links[["n_id","h_id","author_file","text_standardized","link_method","link_score"]],
    on="n_id", how="left"
)

print(f"Master backbone records (Navarro TEI): {len(master_candidate):,}")
print(f"Records enriched with Hernández text: {master_candidate['h_id'].notna().sum():,}")
display(master_candidate[[
    "n_id","author_dir","title","h_id","link_method","link_score"
]].head(15))


## 09. Temporal consequence of the audit

The current data do **not** contain usable poem-level chronology at scale:

- Hernández `Date` = author lifespan.
- The exact TEI date audit finds only bibliographic witness/edition dates, not composition dates.

Therefore the paper cannot honestly assign each sonnet a single year from the existing corpus.

The next phase will construct a **temporal evidence registry** with:

`author / poem-or-collection / date_min / date_max / date_type / confidence / scholarly_source`

The preferred hierarchy remains:

1. composition date or bounded composition interval;
2. first publication / collection circulation date;
3. witness or edition date (bibliographic only);
4. author active/career interval as uncertainty fallback;
5. birth year as covariate only.

A probabilistic/interval temporal design will be considered if exact dating remains sparse.


In [ ]:
print("SCIENTIFIC CHECKPOINT")
print("---------------------")
print("Save this executed notebook to GitHub.")
print("The next decision will be based on:")
print("  1) exact/fuzzy overlap rate between the two corpora;")
print("  2) unresolved segmentation anomalies;")
print("  3) source-exclusive authors/poems (especially Pacheco);")
print("  4) whether Navarro TEI is confirmed as the master poem-level backbone.")
print("\nDo not build semantic networks yet.")
